## Nodes

In [1]:
import itertools
from collections import deque
import pandas as pd
from IPython.display import HTML, display

# the LaTeX in these tables is inline math, so it wraps mid-expression at
# the default (narrow) column width -- keep every cell on one line, and let
# the table scroll horizontally instead of squeezing columns to fit
display(HTML('''<style>
  .dataframe td, .dataframe th { white-space: nowrap; padding: 4px 12px; }
  table.dataframe { display: block; overflow-x: auto; }
</style>'''))

BR = '@@BR@@'   # placeholder for a real <br>, inserted after pandas escapes the cell text

def show(df):
    display(HTML(df.to_html().replace(BR, '<br>')))

CANDS = 'ABC'
ORDERS = list(itertools.permutations(CANDS))       # 6 vote-count orderings, least -> most
PAIRS = [('A', 'B'), ('B', 'C'), ('A', 'C')]        # the 3 pairwise matchups
RANK = {'A': 2, 'B': 1, 'C': 0}                     # coalition preference A > B > C

def tournaments():
    for bits in itertools.product([True, False], repeat=3):
        yield frozenset((x, y) if win else (y, x) for (x, y), win in zip(PAIRS, bits))

TOURNAMENTS = list(tournaments())                   # 8 tournament structures T
NODES = [(order, T) for order in ORDERS for T in TOURNAMENTS]   # P = <order | T>

len(ORDERS), len(TOURNAMENTS), len(NODES)

(6, 8, 48)

In [2]:
def winner(node):
    # f(P): the pairwise winner between the top two vote-getters
    order, T = node
    y, z = order[1], order[2]
    return y if (y, z) in T else z

def fmt_order(order):
    # {<} instead of < : braced, "<" is an ordinary atom instead of a
    # relation, which keeps nbconvert's MathJax (linebreaks.automatic=True)
    # from splitting the formula onto multiple lines at each "<"
    return f"$\\langle {'{<}'.join(order)} \\rangle$"

def fmt_T(T, free=None):
    # X -> Y means X defeats Y; `free`, if given, is the one matchup that
    # never actually got consulted, shown as X <-> Y instead. One matchup
    # per line (BR), since three side by side is hard to scan.
    parts = []
    for x, y in PAIRS:
        if free and {x, y} == set(free):
            parts.append(f'${x} \\leftrightarrow {y}$')
        else:
            a, b = (x, y) if (x, y) in T else (y, x)
            parts.append(f'${a} \\to {b}$')
    return BR.join(parts)

def free_matchup(T1, T2):
    # the one matchup (if any) where T1, T2 disagree
    for x, y in PAIRS:
        if ((x, y) in T1) != ((x, y) in T2):
            return frozenset((x, y))
    return None

## IRV and STAR graphs

In [3]:
def swap(order, i, j):
    lst = list(order)
    lst[i], lst[j] = lst[j], lst[i]
    return tuple(lst)

def irv_moves(node):
    # any adjacent swap in scores, unless it's moving A up
    order, T = node
    out = []
    for i in (0, 1):
        if order[i] == 'A':
            continue
        out.append((swap(order, i, i + 1), T))
    return out

def star_moves(node):
    # any adjacent swap in scores, unless it's moving A above C
    order, T = node
    out = []
    for i in (0, 1):
        j = i + 1
        if order[i] == 'A' and order[j] == 'C':
            continue
        out.append((swap(order, i, j), T))
    return out

irv_graph = {n: irv_moves(n) for n in NODES}
star_graph = {n: star_moves(n) for n in NODES}

sum(len(v) for v in irv_graph.values()), sum(len(v) for v in star_graph.values())

(64, 80)

## Simple manipulations (A≻B≻C coalition)

In [4]:
def mover(node, move):
    # the one candidate whose position changes upward in this swap
    order, new_order = node[0], move[0]
    i, j = [k for k in range(3) if order[k] != new_order[k]]
    return order[i], order[j]   # (who moves up, who it passes)

def move_desc_irv(node, move):
    # every IRV move is the coalition abandoning A -- the mover is who they
    # rank first instead
    up, _ = mover(node, move)
    return f'Betray A for {up}'

def move_desc_star(node, move):
    # every profitable STAR move runs through B's score: 5-4-0 boosts B
    # past whoever was above it, 5-1-0 starves B below whoever passes it
    up, down = mover(node, move)
    if up == 'B':
        return '5-4-0'
    if down == 'B':
        return '5-1-0'
    return f'{up} up past {down}'   # not used by any profitable move here

COLS_SIMPLE = ['start', 'tournament', 'move', 'end', 'f(start)', 'f(end)']

def has_simple_manipulation(node, graph):
    f0 = winner(node)
    return any(RANK[winner(mv)] > RANK[f0] for mv in graph[node])

def simple_manipulations(graph, describe):
    # group by (start order, end order, f(start), f(end)), ignoring T --
    # a group of 2 differs only in one pairwise matchup that never actually
    # got consulted, so we collapse it into a single row and mark that
    # matchup as free (<->) in the tournament column instead of listing
    # both tournament structures separately
    groups = {}
    for node in NODES:
        f0 = winner(node)
        for move in graph[node]:
            f1 = winner(move)
            if RANK[f1] > RANK[f0]:
                key = (node[0], move[0], f0, f1)
                groups.setdefault(key, []).append(node)

    rows = []
    for (order0, order1, f0, f1), nodes in groups.items():
        T0 = nodes[0][1]
        free = free_matchup(T0, nodes[1][1]) if len(nodes) == 2 else None
        node0, move0 = (order0, T0), (order1, T0)
        rows.append({'start': fmt_order(order0), 'tournament': fmt_T(T0, free),
                     'move': describe(node0, move0), 'end': fmt_order(order1),
                     'f(start)': f0, 'f(end)': f1})
    return pd.DataFrame(rows, columns=COLS_SIMPLE)

irv_simple_df = simple_manipulations(irv_graph, move_desc_irv)
show(irv_simple_df)

,start,tournament,move,end,f(start),f(end)
0,$\langle B{<}A{<}C \rangle$,$A \leftrightarrow B$$B \to C$$C \to A$,Betray A for B,$\langle A{<}B{<}C \rangle$,C,B
1,$\langle B{<}C{<}A \rangle$,$A \to B$$B \leftrightarrow C$$C \to A$,Betray A for B,$\langle C{<}B{<}A \rangle$,C,A
2,$\langle B{<}C{<}A \rangle$,$B \to A$$B \leftrightarrow C$$C \to A$,Betray A for B,$\langle C{<}B{<}A \rangle$,C,B
3,$\langle C{<}B{<}A \rangle$,$B \to A$$B \leftrightarrow C$$A \to C$,Betray A for C,$\langle B{<}C{<}A \rangle$,B,A


In [5]:
star_simple_df = simple_manipulations(star_graph, move_desc_star)
show(star_simple_df)

,start,tournament,move,end,f(start),f(end)
0,$\langle A{<}B{<}C \rangle$,$A \leftrightarrow B$$B \to C$$A \to C$,5-1-0,$\langle B{<}A{<}C \rangle$,B,A
1,$\langle A{<}B{<}C \rangle$,$A \leftrightarrow B$$C \to B$$A \to C$,5-1-0,$\langle B{<}A{<}C \rangle$,C,A
2,$\langle B{<}A{<}C \rangle$,$A \leftrightarrow B$$B \to C$$C \to A$,5-4-0,$\langle A{<}B{<}C \rangle$,C,B
3,$\langle B{<}C{<}A \rangle$,$A \to B$$B \leftrightarrow C$$C \to A$,5-4-0,$\langle C{<}B{<}A \rangle$,C,A
4,$\langle B{<}C{<}A \rangle$,$B \to A$$B \leftrightarrow C$$C \to A$,5-4-0,$\langle C{<}B{<}A \rangle$,C,B
5,$\langle C{<}B{<}A \rangle$,$B \to A$$B \leftrightarrow C$$A \to C$,5-1-0,$\langle B{<}C{<}A \rangle$,B,A


## Complex manipulations

In [6]:
def shortest_multistep_path(start, graph):
    # shortest walk of >= 2 safe steps (every node visited weakly preferred,
    # >=, to f(start)) ending at a node strictly preferred to f(start).
    # A direct 1-step (simple) manipulation, if one exists, is ignored here
    # on purpose -- see the `has simple too` column below.
    f0 = winner(start)
    dist = {start: 0}
    prev = {start: None}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        for nxt in graph[cur]:
            if nxt in dist:
                continue
            f1 = winner(nxt)
            if RANK[f1] < RANK[f0]:
                continue              # would regress below the start, not a safe step
            dist[nxt] = dist[cur] + 1
            prev[nxt] = cur
            if RANK[f1] > RANK[f0] and dist[nxt] >= 2:
                path, p = [nxt], cur
                while p is not None:
                    path.append(p)
                    p = prev[p]
                return path[::-1]
            queue.append(nxt)
    return None

COLS_COMPLEX = ['start', 'tournament', 'path', 'steps', 'f(start)', 'f(end)', 'has simple too']

def complex_manipulations(graph):
    # every state with a profitable walk of 2+ safe steps, whether or not a
    # shorter (simple) manipulation is *also* available from it -- grouped
    # and collapsed the same way as simple_manipulations above. T is fixed
    # for the whole walk, so it's shown once rather than repeated per node.
    groups = {}
    for node in NODES:
        path = shortest_multistep_path(node, graph)
        if path is None:
            continue
        f0, f1 = winner(path[0]), winner(path[-1])
        key = (tuple(o for o, _ in path), f0, f1, has_simple_manipulation(node, graph))
        groups.setdefault(key, []).append((node, path))

    rows = []
    for (orders, f0, f1, hs), members in groups.items():
        T0 = members[0][0][1]
        free = free_matchup(T0, members[1][0][1]) if len(members) == 2 else None
        rows.append({'start': fmt_order(orders[0]), 'tournament': fmt_T(T0, free),
                     'path': ' → '.join(fmt_order(o) for o in orders),
                     'steps': len(orders) - 1,
                     'f(start)': f0, 'f(end)': f1, 'has simple too': hs})
    return pd.DataFrame(rows, columns=COLS_COMPLEX)

irv_complex_df = complex_manipulations(irv_graph)
show(irv_complex_df)

,start,tournament,path,steps,f(start),f(end),has simple too
0,$\langle B{<}A{<}C \rangle$,$A \leftrightarrow B$$B \to C$$C \to A$,$\langle B{<}A{<}C \rangle$ → $\langle A{<}B{<}C \rangle$ → $\langle A{<}C{<}B \rangle$,2,C,B,True
1,$\langle B{<}C{<}A \rangle$,$A \to B$$B \leftrightarrow C$$C \to A$,$\langle B{<}C{<}A \rangle$ → $\langle C{<}B{<}A \rangle$ → $\langle C{<}A{<}B \rangle$,2,C,A,True
2,$\langle B{<}C{<}A \rangle$,$B \to A$$B \leftrightarrow C$$C \to A$,$\langle B{<}C{<}A \rangle$ → $\langle C{<}B{<}A \rangle$ → $\langle C{<}A{<}B \rangle$,2,C,B,True
3,$\langle C{<}B{<}A \rangle$,$B \to A$$B \leftrightarrow C$$A \to C$,$\langle C{<}B{<}A \rangle$ → $\langle B{<}C{<}A \rangle$ → $\langle B{<}A{<}C \rangle$,2,B,A,True


In [7]:
star_complex_df = complex_manipulations(star_graph)
show(star_complex_df)

,start,tournament,path,steps,f(start),f(end),has simple too
0,$\langle A{<}C{<}B \rangle$,$A \leftrightarrow B$$B \to C$$A \to C$,$\langle A{<}C{<}B \rangle$ → $\langle A{<}B{<}C \rangle$ → $\langle B{<}A{<}C \rangle$,2,B,A,False
1,$\langle A{<}C{<}B \rangle$,$A \leftrightarrow B$$C \to B$$A \to C$,$\langle A{<}C{<}B \rangle$ → $\langle A{<}B{<}C \rangle$ → $\langle B{<}A{<}C \rangle$,2,C,A,False
2,$\langle B{<}A{<}C \rangle$,$A \leftrightarrow B$$B \to C$$C \to A$,$\langle B{<}A{<}C \rangle$ → $\langle A{<}B{<}C \rangle$ → $\langle A{<}C{<}B \rangle$,2,C,B,True
3,$\langle B{<}C{<}A \rangle$,$A \to B$$B \leftrightarrow C$$C \to A$,$\langle B{<}C{<}A \rangle$ → $\langle C{<}B{<}A \rangle$ → $\langle C{<}A{<}B \rangle$,2,C,A,True
4,$\langle B{<}C{<}A \rangle$,$B \to A$$B \leftrightarrow C$$C \to A$,$\langle B{<}C{<}A \rangle$ → $\langle C{<}B{<}A \rangle$ → $\langle C{<}A{<}B \rangle$,2,C,B,True
5,$\langle C{<}A{<}B \rangle$,$B \to A$$B \leftrightarrow C$$A \to C$,$\langle C{<}A{<}B \rangle$ → $\langle C{<}B{<}A \rangle$ → $\langle B{<}C{<}A \rangle$,2,B,A,False
6,$\langle C{<}B{<}A \rangle$,$B \to A$$B \leftrightarrow C$$A \to C$,$\langle C{<}B{<}A \rangle$ → $\langle B{<}C{<}A \rangle$ → $\langle B{<}A{<}C \rangle$,2,B,A,True


## Total manipulable states

In [8]:
# a state is manipulable if it has a simple manipulation, a complex one, or both
def is_manipulable(node, graph):
    return has_simple_manipulation(node, graph) or shortest_multistep_path(node, graph) is not None

irv_manipulable = {n for n in NODES if is_manipulable(n, irv_graph)}
star_manipulable = {n for n in NODES if is_manipulable(n, star_graph)}

# cross-check against the two tables above: each row stands for 2 states if
# its tournament column has a free (<->) matchup, 1 otherwise; complex rows
# with `has simple too` would double-count a state already covered by the
# simple table, so skip those
def table_count(df, skip_has_simple=False):
    rows = df[~df['has simple too']] if skip_has_simple and 'has simple too' in df else df
    return sum(2 if 'leftrightarrow' in t else 1 for t in rows['tournament'])

irv_from_tables = table_count(irv_simple_df) + table_count(irv_complex_df, skip_has_simple=True)
star_from_tables = table_count(star_simple_df) + table_count(star_complex_df, skip_has_simple=True)
assert irv_from_tables == len(irv_manipulable)
assert star_from_tables == len(star_manipulable)

pd.Series({'IRV': len(irv_manipulable), 'STAR': len(star_manipulable)}, name='manipulable states (of 48)')

IRV      8
STAR    18
Name: manipulable states (of 48), dtype: int64